# 04 · Publish dashboard data
Exports the latest gold-layer results as one JSON file and commits it to the GitHub repository,
where the live dashboard on GitHub Pages reads it.

- Reads **only gold tables and `dq_results`**, so the dashboard shows exactly what the pipeline verified
- Runs **after** `03_data_quality`, so the latest check results are included, including any failures
- The GitHub token is read from the config volume at runtime and never written in code
- The browser never connects to Databricks: it only reads a static JSON file, so no credentials are ever exposed publicly

In [0]:
import base64, json
from datetime import datetime, timezone
from decimal import Decimal

import requests

SCHEMA = "workspace.media_pipeline"
REPO = "BalamuruganS-Data/media-campaign-pipeline"
TARGET_PATH = "docs/data/dashboard.json"
BRANCH = "main"
TOKEN_FILE = "/Volumes/workspace/media_pipeline/config/github_token.txt"

spark.sql("SET TIME ZONE 'UTC'")

def rows(sql):
    """Run a query and return JSON-safe dictionaries (decimals become floats)."""
    out = []
    for r in spark.sql(sql).collect():
        d = r.asDict()
        out.append({k: float(v) if isinstance(v, Decimal) else v for k, v in d.items()})
    return out

## 1. Build the export from the gold layer
Platform totals keep `spend_complete` separately, so the dashboard can apply the same rules as the SQL view and Power BI:
ratios from totals, incomplete-day spend excluded from cost ratios, and CPA only from platforms that report conversions.

In [0]:
export = {
    "generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),

    "platforms": rows(f"""
        SELECT p.platform,
               p.platform_name                                     AS name,
               SUM(f.spend)                                        AS spend,
               SUM(f.spend) FILTER (WHERE NOT f.is_incomplete)     AS spend_complete,
               SUM(f.impressions)                                  AS impressions,
               SUM(f.clicks)                                       AS clicks,
               SUM(f.conversions)                                  AS conversions
        FROM {SCHEMA}.gold_fact_campaign_performance f
        JOIN {SCHEMA}.gold_dim_platform p ON f.platform_key = p.platform_key
        GROUP BY p.platform, p.platform_name
    """),

    "daily": rows(f"""
        SELECT c.campaign_id                          AS campaign,
               date_format(d.full_date, 'yyyy-MM-dd') AS date,
               f.spend, f.impressions, f.clicks, f.conversions
        FROM {SCHEMA}.gold_fact_campaign_performance f
        JOIN {SCHEMA}.gold_dim_platform p ON f.platform_key = p.platform_key
        JOIN {SCHEMA}.gold_dim_campaign c ON f.campaign_key = c.campaign_key
        JOIN {SCHEMA}.gold_dim_date d     ON f.date_key = d.date_key
        WHERE p.platform = 'daily_campaign_platform' AND f.date_key <> 0
        ORDER BY campaign, date
    """),

    "multichannel_monthly": rows(f"""
        SELECT date_format(d.full_date, 'yyyy-MM') AS month,
               SUM(f.spend)                        AS spend
        FROM {SCHEMA}.gold_fact_campaign_performance f
        JOIN {SCHEMA}.gold_dim_platform p ON f.platform_key = p.platform_key
        JOIN {SCHEMA}.gold_dim_date d     ON f.date_key = d.date_key
        WHERE p.platform = 'multichannel_platform' AND f.date_key <> 0
        GROUP BY 1
        ORDER BY 1
    """),

    "reconciliation": rows(f"""
        SELECT platform, billing_month, delivered_spend, billed_amount, variance, has_duplicate_invoice, status
        FROM {SCHEMA}.gold_vw_billing_reconciliation
        ORDER BY platform, billing_month
    """),

    # Market, client and media-channel breakdowns (multichannel only; other sources have no such attributes)
    "breakdowns": rows(f"""
        SELECT date_format(d.full_date, 'yyyy-MM') AS month,
               c.market, c.client, c.media_channel,
               SUM(f.spend)       AS spend,
               SUM(f.impressions) AS impressions,
               SUM(f.clicks)      AS clicks,
               COUNT(*)           AS campaigns
        FROM {SCHEMA}.gold_fact_campaign_performance f
        JOIN {SCHEMA}.gold_dim_platform p ON f.platform_key = p.platform_key
        JOIN {SCHEMA}.gold_dim_campaign c ON f.campaign_key = c.campaign_key
        JOIN {SCHEMA}.gold_dim_date d     ON f.date_key = d.date_key
        WHERE p.platform = 'multichannel_platform' AND f.date_key <> 0
        GROUP BY 1, 2, 3, 4
        ORDER BY 1, 2, 3, 4
    """),

    "dq_results": rows(f"""
        SELECT date_format(run_at, "yyyy-MM-dd'T'HH:mm:ss'Z'") AS run_at,
               check_name, layer, expected, actual, status
        FROM {SCHEMA}.dq_results
        ORDER BY run_at, check_name
    """),
}

for key, value in export.items():
    print(f"{key}: {len(value) if isinstance(value, list) else value}")

## 2. Validate before publishing
Never publish an export that doesn't reconcile with the fact table. A failed assertion stops the notebook,
so the live dashboard keeps showing the last good data instead of a broken one.

In [0]:
fact_total = float(spark.sql(f"SELECT SUM(spend) FROM {SCHEMA}.gold_fact_campaign_performance").first()[0])
export_total = sum(p["spend"] for p in export["platforms"])

assert export["platforms"], "No platform rows to publish"
assert abs(fact_total - export_total) < 0.01, f"Export spend {export_total} does not match fact spend {fact_total}"
assert export["dq_results"], "No data quality results to publish"

multi_spend = next(p["spend"] for p in export["platforms"] if p["platform"] == "multichannel_platform")
breakdown_spend = sum(r["spend"] for r in export["breakdowns"])
assert abs(multi_spend - breakdown_spend) < 0.01, f"Breakdown spend {breakdown_spend} does not match multichannel spend {multi_spend}"

print(f"Validated: export spend {export_total:,.2f} matches fact spend {fact_total:,.2f}")

## 3. Commit the export to GitHub
Uses the GitHub contents API. Updating an existing file requires its current `sha`, which acts as a version check
so a stale write can't overwrite a newer one.

In [0]:
with open(TOKEN_FILE) as f:
    token = f.read().strip()

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
url = f"https://api.github.com/repos/{REPO}/contents/{TARGET_PATH}"

existing = requests.get(url, headers=headers, params={"ref": BRANCH}, timeout=30)
sha = existing.json().get("sha") if existing.status_code == 200 else None

body = {
    "message": f"Refresh dashboard data {export['generated_at']}",
    "content": base64.b64encode(json.dumps(export, indent=1, default=str).encode("utf-8")).decode("ascii"),
    "branch": BRANCH,
}
if sha:
    body["sha"] = sha

response = requests.put(url, headers=headers, json=body, timeout=30)
response.raise_for_status()
print("Published:", response.json()["commit"]["html_url"])